# Day 13: Capstone Project — Build Your Own Production Agent 🏆

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**This notebook is a guided template. You fill in the TODO sections.**

Your agent must demonstrate all 6 mandatory capabilities:
1. ✅ LangGraph StateGraph (3+ nodes)
2. ✅ ChromaDB RAG (10+ documents)
3. ✅ Conversation memory (MemorySaver + thread_id)
4. ✅ Self-reflection (eval node or review loop)
5. ✅ Tool use (at least one tool beyond retrieval)
6. ✅ Deployment (Streamlit UI or FastAPI)

---
### Before you write any code — answer these three questions:
1. **What domain am I building for?** (e.g., HR Policy Bot, Study Buddy for Physics, Research Assistant)
2. **Who is the user?** (e.g., students asking questions, employees checking policies)
3. **What does success look like?** (e.g., agent answers 90% of domain questions faithfully)

Write your answers in the cell below before proceeding.

## My Capstone Plan

**Domain:** Python Coding Assistant — an agent that helps learners understand Python concepts, debug code, and follow best practices.

**User:** Beginner to intermediate Python learners, bootcamp students, and self-taught developers who have quick coding questions.

**Success looks like:** The agent accurately answers ≥ 90% of Python questions from its knowledge base, correctly admits when a topic is out of scope, and provides web-searched answers for questions about current library versions or recent changes.

**Tool I will add:** Web search (DuckDuckGo via duckduckgo-search) — useful when a user asks about a library's latest version, a recently released feature, or real-world package documentation.

**Deployment choice:** Streamlit UI

---
## 0. Setup

In [3]:
!pip install langgraph langchain-groq langchain-core chromadb \
              sentence-transformers ragas ddgs python-dotenv duckduckgo-search -q

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = "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"


In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, List
import chromadb
from sentence_transformers import SentenceTransformer
from importlib.metadata import version

groq_key = os.getenv("GROQ_API_KEY", "")
print(f"Groq API Key: {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing'}")
print(f"LangGraph:    {version('langgraph')}")

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
r = llm.invoke("Say ready in 1 word.")
print(f"LLM:          ✅ {r.content}")

Groq API Key: ✅ Loaded
LangGraph:    1.1.6
LLM:          ✅ Ready.


---
## Part 1 — Domain Setup: Knowledge Base

Load at least 10 documents about your domain. Write them as strings or load from files.

**Tips:**
- Each document should be 100-500 words
- Cover different aspects of your domain (don't repeat the same topic)
- Documents should be specific enough to answer concrete questions

In [5]:
DOCUMENTS = [
    {
        "id": "doc_001",
        "topic": "Python Lists",
        "text": """A list in Python is an ordered, mutable collection that can hold items of any type.
You create a list with square brackets: my_list = [1, 2, 3].
Common operations:
- Append: my_list.append(4)  — adds to the end, O(1).
- Insert: my_list.insert(0, 0)  — inserts at index, O(n).
- Remove: my_list.remove(2)  — removes first occurrence by value.
- Pop: my_list.pop()  — removes and returns last item; pop(i) removes index i.
- Slicing: my_list[1:3] returns a new list with elements at index 1 and 2.
- List comprehension: squares = [x**2 for x in range(10)] is the Pythonic way to build lists.
Lists are zero-indexed. Negative indices count from the end: my_list[-1] is the last element.
len(my_list) returns the number of elements. Use 'in' to check membership: 3 in my_list.
Sorting: my_list.sort() sorts in-place; sorted(my_list) returns a new sorted list."""
    },
    {
        "id": "doc_002",
        "topic": "Python Dictionaries",
        "text": """A dictionary (dict) is an unordered collection of key-value pairs.
Keys must be hashable (strings, numbers, tuples). Values can be any type.
Creation: person = {"name": "Alice", "age": 30}
Access: person["name"] returns "Alice". Use person.get("city", "Unknown") for a safe default.
Adding/updating: person["email"] = "alice@example.com"
Deleting: del person["age"] or person.pop("age")
Iterating: for key, value in person.items(): loops over pairs.
Keys only: for key in person or person.keys()
Values only: person.values()
Check membership: "name" in person  — checks keys by default, O(1).
Dict comprehension: squares = {x: x**2 for x in range(5)}
Merging (Python 3.9+): merged = dict1 | dict2
As of Python 3.7+ dictionaries maintain insertion order."""
    },
    {
        "id": "doc_003",
        "topic": "Python Functions and Scope",
        "text": """Functions are defined with the 'def' keyword:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"

Key concepts:
- Positional and keyword arguments: greet("Alice") or greet(name="Alice", greeting="Hi")
- *args collects extra positional args as a tuple.
- **kwargs collects extra keyword args as a dict.
- Default parameter values are evaluated once at definition time — never use mutable defaults like [].
  Use None as default and set inside the body instead.
- Return multiple values: return x, y  — Python packs them as a tuple.
Scope rules (LEGB): Local -> Enclosing -> Global -> Built-in.
- 'global x' inside a function lets you modify a module-level variable.
- 'nonlocal x' inside a nested function lets you modify the enclosing function's variable.
Lambda functions: square = lambda x: x**2  — single-expression anonymous functions.
Docstrings: the first string literal inside a function becomes its __doc__ attribute."""
    },
    {
        "id": "doc_004",
        "topic": "Python Classes and OOP",
        "text": """Python is an object-oriented language. Classes define blueprints for objects.
class Animal:
    def __init__(self, name, sound):
        self.name = name
        self.sound = sound

    def speak(self):
        return f"{self.name} says {self.sound}"

Inheritance: class Dog(Animal):  — Dog inherits all of Animal's methods.
Call parent constructor: super().__init__(name, "Woof")
Key dunder (magic) methods:
- __str__: controls str(obj) output
- __repr__: controls repr(obj), should be unambiguous
- __len__: makes len(obj) work
- __eq__: defines == behavior
- __add__: defines + behavior
Class variables are shared across instances; instance variables are unique per object.
@property decorator turns a method into a read-only attribute.
@classmethod receives the class (cls) instead of an instance; used for alternative constructors.
@staticmethod receives neither; it is just a namespaced function."""
    },
    {
        "id": "doc_005",
        "topic": "Exception Handling",
        "text": """Python uses try/except blocks to handle errors gracefully.
try:
    result = 10 / divisor
except ZeroDivisionError:
    print("Cannot divide by zero")
except (TypeError, ValueError) as e:
    print(f"Bad input: {e}")
else:
    print("Success:", result)   # runs only if no exception occurred
finally:
    print("Always runs")        # good for cleanup (closing files, DB connections)

Raise your own exceptions: raise ValueError("Age must be positive")
Create custom exceptions by subclassing Exception:
class InsufficientFundsError(Exception):
    pass
Exception hierarchy: BaseException -> Exception -> specific errors.
Use 'except Exception' to catch almost anything (avoids SystemExit, KeyboardInterrupt).
Context managers with 'with' handle resource cleanup automatically:
with open("file.txt") as f:
    data = f.read()  # file closes automatically even if an error occurs"""
    },
    {
        "id": "doc_006",
        "topic": "Python List Comprehensions and Generators",
        "text": """List comprehensions provide a concise way to build lists:
evens = [x for x in range(20) if x % 2 == 0]
nested = [x*y for x in range(3) for y in range(3)]

Dict and set comprehensions work the same way:
square_map = {x: x**2 for x in range(5)}
unique_chars = {c for c in "hello world" if c != " "}

Generator expressions are like list comprehensions but lazy — they produce values one at a time:
gen = (x**2 for x in range(1000000))  # uses almost no memory
next(gen) retrieves the next value.

Generator functions use 'yield':
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

Use itertools for advanced iteration: chain, islice, product, combinations, permutations.
Prefer generators over large lists when you only need to iterate once — they save memory."""
    },
    {
        "id": "doc_007",
        "topic": "File I/O",
        "text": """Reading and writing files in Python:
# Read entire file
with open("data.txt", "r", encoding="utf-8") as f:
    content = f.read()

# Read line by line (memory-efficient)
with open("data.txt") as f:
    for line in f:
        process(line.rstrip())

# Write a file (overwrites)
with open("output.txt", "w") as f:
    f.write("Hello\n")

# Append
with open("log.txt", "a") as f:
    f.write("New entry\n")

File modes: "r" read, "w" write (overwrite), "a" append, "b" binary, "r+" read+write.
Always use 'with' — it guarantees the file is closed even on errors.
pathlib.Path is the modern way to work with file paths:
from pathlib import Path
p = Path("data") / "file.txt"
p.read_text()   # reads content
p.write_text("hello")
p.exists(), p.is_file(), p.is_dir(), p.suffix, p.stem, p.parent"""
    },
    {
        "id": "doc_008",
        "topic": "Python Decorators",
        "text": """A decorator is a function that wraps another function to extend its behavior.
def timer(func):
    import time
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        print(f"{func.__name__} took {time.time()-start:.4f}s")
        return result
    return wrapper

@timer
def slow_function():
    time.sleep(1)

The @timer syntax is shorthand for slow_function = timer(slow_function).
Use functools.wraps to preserve the wrapped function's metadata:
from functools import wraps
def decorator(func):
    @wraps(func)  # preserves __name__, __doc__, etc.
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

Decorators with arguments require an extra layer:
def repeat(n):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(n):
                func(*args, **kwargs)
        return wrapper
    return decorator

@repeat(3)
def say_hello(): print("Hello")"""
    },
    {
        "id": "doc_009",
        "topic": "Virtual Environments and pip",
        "text": """A virtual environment isolates a project's dependencies from the global Python install.
Creating and activating:
python -m venv venv          # creates folder 'venv'
source venv/bin/activate     # macOS/Linux
venv\\Scripts\\activate       # Windows
deactivate                   # leave the environment

Installing packages:
pip install requests         # latest version
pip install requests==2.31.0 # specific version
pip install -r requirements.txt  # install from file

Saving dependencies:
pip freeze > requirements.txt    # export current environment

Modern tools:
- pipenv: combines pip + venv, uses Pipfile
- poetry: advanced dependency management with pyproject.toml
- uv: ultra-fast Rust-based installer (2024)

Always add venv/ (or .venv/) to your .gitignore.
Use python -m pip instead of bare pip to ensure you are using the right Python interpreter.
Check installed packages: pip list or pip show requests"""
    },
    {
        "id": "doc_010",
        "topic": "Debugging and Testing",
        "text": """Debugging Python code:
- print() debugging: quick but messy.
- pdb (Python Debugger): import pdb; pdb.set_trace()  — sets a breakpoint.
  Python 3.7+: breakpoint()  — shorthand for pdb.set_trace().
  Commands: n (next), s (step into), c (continue), q (quit), p var (print variable).
- IDE debuggers (VS Code, PyCharm) provide visual breakpoints.

Unit testing with pytest:
def add(a, b): return a + b

def test_add():
    assert add(2, 3) == 5
    assert add(-1, 1) == 0

Run: pytest test_file.py -v
pytest discovers any function starting with 'test_'.
Fixtures provide reusable setup:
@pytest.fixture
def sample_list(): return [1, 2, 3]

def test_length(sample_list):
    assert len(sample_list) == 3

Coverage: pytest --cov=mymodule to see what percentage of code is tested.
unittest is the built-in alternative — less concise than pytest but no extra install needed."""
    },
    {
        "id": "doc_011",
        "topic": "Python Type Hints",
        "text": """Type hints (PEP 484) let you annotate variable and function types for better IDE support and static analysis.
def greet(name: str, age: int) -> str:
    return f"Hello {name}, you are {age}"

Common types from the typing module (Python 3.9+ — many built-ins work directly):
from typing import List, Dict, Tuple, Optional, Union, Any

# Python 3.9+ shorthand:
def process(items: list[int]) -> dict[str, int]: ...

Optional[str] means str or None — equivalent to str | None (Python 3.10+).
Union[int, str] means either type — Python 3.10+: int | str.

Type hints are NOT enforced at runtime — they are hints only.
Use mypy for static type checking: mypy script.py
TypeVar for generic functions:
from typing import TypeVar
T = TypeVar("T")
def first(items: list[T]) -> T:
    return items[0]

dataclasses use type hints to auto-generate __init__, __repr__, __eq__:
from dataclasses import dataclass
@dataclass
class Point:
    x: float
    y: float"""
    },
    {
        "id": "doc_012",
        "topic": "Common Python Pitfalls",
        "text": """Mutable default arguments — the most common Python bug:
# WRONG:
def append_to(element, to=[]):
    to.append(element)
    return to
# to=[] is created once; every call shares the same list.

# CORRECT:
def append_to(element, to=None):
    if to is None:
        to = []
    to.append(element)
    return to

Late binding in closures:
funcs = [lambda: i for i in range(5)]
funcs[0]()  # returns 4, not 0 — all lambdas share the same i
Fix: lambda i=i: i  — captures current value.

Integer identity vs equality:
a = 256; b = 256; a is b  # True (CPython caches -5 to 256)
a = 1000; b = 1000; a is b  # False — do not use 'is' for value comparison.

Circular imports: modules that import each other can cause ImportError.
Fix: move imports inside functions or restructure the package.

Forgetting to copy: new_list = old_list assigns the same reference.
Use new_list = old_list.copy() or list(old_list) for a shallow copy.
Use copy.deepcopy(obj) when nested objects must also be copied."""
    },
]

# ── Build ChromaDB ─────────────────────────────────────────
print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.Client()
try:
    client.delete_collection("capstone_kb")
except:
    pass
collection = client.create_collection("capstone_kb")

texts = [d["text"] for d in DOCUMENTS]
ids   = [d["id"]   for d in DOCUMENTS]
embeddings = embedder.encode(texts).tolist()

collection.add(
    documents=texts,
    embeddings=embeddings,
    ids=ids,
    metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
)

print(f"✅ Knowledge base ready: {collection.count()} documents")
for d in DOCUMENTS:
    print(f"   • {d['topic']}")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Knowledge base ready: 12 documents
   • Python Lists
   • Python Dictionaries
   • Python Functions and Scope
   • Python Classes and OOP
   • Exception Handling
   • Python List Comprehensions and Generators
   • File I/O
   • Python Decorators
   • Virtual Environments and pip
   • Debugging and Testing
   • Python Type Hints
   • Common Python Pitfalls


In [6]:
test_query = "How do I sort a list in Python?"

q_emb   = embedder.encode([test_query]).tolist()
results = collection.query(query_embeddings=q_emb, n_results=3)

print(f"Query: {test_query}")
print(f"\nTop 3 retrieved chunks:")
for i, (doc, meta) in enumerate(zip(results["documents"][0], results["metadatas"][0])):
    print(f"\n[{i+1}] Topic: {meta['topic']}")
    print(f"    Text: {doc[:200]}...")

print("\n✅ If the retrieved chunks are relevant — retrieval is working correctly.")

Query: How do I sort a list in Python?

Top 3 retrieved chunks:

[1] Topic: Python Lists
    Text: A list in Python is an ordered, mutable collection that can hold items of any type.
You create a list with square brackets: my_list = [1, 2, 3].
Common operations:
- Append: my_list.append(4)  — adds ...

[2] Topic: Python Type Hints
    Text: Type hints (PEP 484) let you annotate variable and function types for better IDE support and static analysis.
def greet(name: str, age: int) -> str:
    return f"Hello {name}, you are {age}"

Common t...

[3] Topic: Python Dictionaries
    Text: A dictionary (dict) is an unordered collection of key-value pairs.
Keys must be hashable (strings, numbers, tuples). Values can be any type.
Creation: person = {"name": "Alice", "age": 30}
Access: per...

✅ If the retrieved chunks are relevant — retrieval is working correctly.


---
## Part 2 — State Design

**Design your State TypedDict BEFORE writing any node.** Every field a node needs must be a State field.

The template below is the base. Add domain-specific fields as needed.

In [7]:
class CapstoneState(TypedDict):
    # ── Input ──────────────────────────────────────────────
    question:       str

    # ── Memory ─────────────────────────────────────────────
    messages:       List[dict]

    # ── Routing ────────────────────────────────────────────
    route:          str          # "retrieve", "memory_only", "tool"

    # ── RAG ────────────────────────────────────────────────
    retrieved:      str
    sources:        List[str]

    # ── Tool ───────────────────────────────────────────────
    tool_result:    str          # web search result

    # ── Answer ─────────────────────────────────────────────
    answer:         str

    # ── Quality control ────────────────────────────────────
    faithfulness:   float
    eval_retries:   int

    # ── Domain-specific ────────────────────────────────────
    search_results: str          # raw web search output before formatting

print("State defined with fields:", list(CapstoneState.__annotations__.keys()))

State defined with fields: ['question', 'messages', 'route', 'retrieved', 'sources', 'tool_result', 'answer', 'faithfulness', 'eval_retries', 'search_results']


---
## Part 3 — Node Functions

Write each node as a Python function. **Test each node in isolation before adding it to the graph.**

The mandatory nodes are scaffolded below. Add domain-specific nodes as needed.

In [8]:
def memory_node(state: CapstoneState) -> dict:
    msgs = state.get("messages", [])
    msgs = msgs + [{"role": "user", "content": state["question"]}]
    if len(msgs) > 6:  # sliding window: keep last 3 turns
        msgs = msgs[-6:]
    return {"messages": msgs}

# Quick test
test_state = {"question": "What is a Python list?", "messages": []}
result = memory_node(test_state)
print(f"memory_node test: messages={result['messages']}")
print("✅ memory_node works")

memory_node test: messages=[{'role': 'user', 'content': 'What is a Python list?'}]
✅ memory_node works


In [9]:
def router_node(state: CapstoneState) -> dict:
    question = state["question"]
    messages = state.get("messages", [])
    recent   = "; ".join(f"{m['role']}: {m['content'][:60]}" for m in messages[-3:-1]) or "none"

    prompt = f"""You are a router for a Python programming assistant chatbot.

Available options:
- retrieve: search the knowledge base for Python concepts, syntax, built-ins, OOP, decorators, testing, etc.
- memory_only: answer from conversation history (e.g. 'what did you just say?', 'can you repeat that?', 'what was the example you gave?')
- tool: use the web_search tool (e.g. latest library version, recent Python release notes, package documentation URL, real-time PyPI info)

Recent conversation: {recent}
Current question: {question}

Reply with ONLY one word: retrieve / memory_only / tool"""

    response = llm.invoke(prompt)
    decision = response.content.strip().lower()

    if "memory" in decision:   decision = "memory_only"
    elif "tool" in decision:   decision = "tool"
    else:                      decision = "retrieve"

    return {"route": decision}

# Quick test
test_state2 = {"question": "What did you just say?", "messages": [{"role": "user", "content": "hi"}]}
result2 = router_node(test_state2)
print(f"router_node test: route='{result2['route']}' (expected: memory_only)")

router_node test: route='memory_only' (expected: memory_only)


In [10]:
def retrieval_node(state: CapstoneState) -> dict:
    q_emb   = embedder.encode([state["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    topics  = [m["topic"] for m in results["metadatas"][0]]
    context = "\n\n---\n\n".join(f"[{topics[i]}]\n{chunks[i]}" for i in range(len(chunks)))
    return {"retrieved": context, "sources": topics}


def skip_retrieval_node(state: CapstoneState) -> dict:
    return {"retrieved": "", "sources": []}


# Quick test
test_state3 = {"question": "How do I add an item to a list in Python?"}
result3 = retrieval_node(test_state3)
print(f"retrieval_node test: sources={result3['sources']}")
print(f"  Context preview: {result3['retrieved'][:200]}...")
print("✅ retrieval_node works")

retrieval_node test: sources=['Python Lists', 'Common Python Pitfalls', 'Python Type Hints']
  Context preview: [Python Lists]
A list in Python is an ordered, mutable collection that can hold items of any type.
You create a list with square brackets: my_list = [1, 2, 3].
Common operations:
- Append: my_list.app...
✅ retrieval_node works


In [11]:
def tool_node(state: CapstoneState) -> dict:
    """Web search for current Python info (library versions, recent releases, docs)."""
    question = state["question"]

    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(question + " Python", max_results=4))
        if results:
            tool_result = "\n\n".join(
                f"Source: {r['href']}\nTitle: {r['title']}\nSnippet: {r['body'][:300]}"
                for r in results
            )
        else:
            tool_result = "No web results found for this query."
    except Exception as e:
        tool_result = f"Web search unavailable: {e}"

    return {"tool_result": tool_result}

print("tool_node defined — uses DuckDuckGo web search")
print("Install if needed: pip install duckduckgo-search")

tool_node defined — uses DuckDuckGo web search
Install if needed: pip install duckduckgo-search


In [12]:
def answer_node(state: CapstoneState) -> dict:
    question     = state["question"]
    retrieved    = state.get("retrieved", "")
    tool_result  = state.get("tool_result", "")
    messages     = state.get("messages", [])
    eval_retries = state.get("eval_retries", 0)

    context_parts = []
    if retrieved:
        context_parts.append(f"KNOWLEDGE BASE:\n{retrieved}")
    if tool_result:
        context_parts.append(f"TOOL RESULT:\n{tool_result}")
    context = "\n\n".join(context_parts)

    if context:
        system_content = f"""You are a friendly Python programming assistant helping learners understand Python clearly.
Answer using ONLY the information provided in the context below.
Use concrete code examples whenever possible.
If the answer is not in the context, say: "I don't have that in my knowledge base — try asking about lists, dicts, functions, classes, exceptions, comprehensions, file I/O, decorators, environments, debugging, type hints, or common pitfalls."
Do NOT fabricate Python behaviour not stated in the context.

{context}"""
    else:
        system_content = """You are a friendly Python programming assistant.
Answer based on the conversation history. Be concise and use code examples."""

    if eval_retries > 0:
        system_content += "\n\nIMPORTANT: Your previous answer did not meet quality standards. Answer using ONLY information explicitly stated in the context above."

    lc_msgs = [SystemMessage(content=system_content)]
    for msg in messages[:-1]:
        lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user"
                       else AIMessage(content=msg["content"]))
    lc_msgs.append(HumanMessage(content=question))

    response = llm.invoke(lc_msgs)
    return {"answer": response.content}

print("answer_node defined")

answer_node defined


In [13]:
FAITHFULNESS_THRESHOLD = 0.7
MAX_EVAL_RETRIES       = 2

def eval_node(state: CapstoneState) -> dict:
    answer   = state.get("answer", "")
    context  = state.get("retrieved", "")[:500]
    retries  = state.get("eval_retries", 0)

    if not context:
        return {"faithfulness": 1.0, "eval_retries": retries + 1}

    prompt = f"""Rate faithfulness: does this answer use ONLY information from the context?
Reply with ONLY a number between 0.0 and 1.0.
1.0 = fully faithful. 0.5 = some hallucination. 0.0 = mostly hallucinated.

Context: {context}
Answer: {answer[:300]}"""

    result = llm.invoke(prompt).content.strip()
    try:
        score = float(result.split()[0].replace(",", "."))
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5

    gate = "✅" if score >= FAITHFULNESS_THRESHOLD else "⚠️"
    print(f"  [eval] Faithfulness: {score:.2f} {gate}")
    return {"faithfulness": score, "eval_retries": retries + 1}


def save_node(state: CapstoneState) -> dict:
    messages = state.get("messages", [])
    messages = messages + [{"role": "assistant", "content": state["answer"]}]
    return {"messages": messages}

print("eval_node and save_node defined")

eval_node and save_node defined


---
## Part 4 — Graph Assembly

Connect your nodes. The routing functions decide which path to take.

In [14]:
def route_decision(state: CapstoneState) -> str:
    route = state.get("route", "retrieve")
    if route == "tool":        return "tool"
    if route == "memory_only": return "skip"
    return "retrieve"


def eval_decision(state: CapstoneState) -> str:
    score   = state.get("faithfulness", 1.0)
    retries = state.get("eval_retries", 0)
    if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES:
        return "save"
    return "answer"


graph = StateGraph(CapstoneState)

graph.add_node("memory",   memory_node)
graph.add_node("router",   router_node)
graph.add_node("retrieve", retrieval_node)
graph.add_node("skip",     skip_retrieval_node)
graph.add_node("tool",     tool_node)
graph.add_node("answer",   answer_node)
graph.add_node("eval",     eval_node)
graph.add_node("save",     save_node)

graph.set_entry_point("memory")
graph.add_edge("memory", "router")

graph.add_conditional_edges(
    "router", route_decision,
    {"retrieve": "retrieve", "skip": "skip", "tool": "tool"}
)

graph.add_edge("retrieve", "answer")
graph.add_edge("skip",     "answer")
graph.add_edge("tool",     "answer")
graph.add_edge("answer",   "eval")

graph.add_conditional_edges(
    "eval", eval_decision,
    {"answer": "answer", "save": "save"}
)
graph.add_edge("save", END)

checkpointer = MemorySaver()
app = graph.compile(checkpointer=checkpointer)

print("✅ Graph compiled successfully!")
print("Nodes:", ["memory", "router", "retrieve/skip/tool", "answer", "eval", "save"])

✅ Graph compiled successfully!
Nodes: ['memory', 'router', 'retrieve/skip/tool', 'answer', 'eval', 'save']


---
## Part 5 — Testing

Test with at least 10 questions including 2 red-team tests. Document each as PASS or FAIL.

In [15]:
def ask(question: str, thread_id: str = "test") -> dict:
    config = {"configurable": {"thread_id": thread_id}}
    result = app.invoke({"question": question}, config=config)
    return result


TEST_QUESTIONS = [
    {"q": "How do I append an item to a Python list?",                         "expect": "Should explain append() from KB",                      "red_team": False},
    {"q": "What is the difference between a list and a dictionary in Python?", "expect": "Should contrast both data structures from KB",          "red_team": False},
    {"q": "How do I handle a ZeroDivisionError in Python?",                    "expect": "Should explain try/except from KB",                    "red_team": False},
    {"q": "What is a list comprehension and how is it different from a loop?", "expect": "Should explain comprehension syntax from KB",           "red_team": False},
    {"q": "How do I read a file line by line in Python?",                      "expect": "Should mention 'with open' and for loop from KB",      "red_team": False},
    {"q": "What does the @wraps decorator do?",                                "expect": "Should explain functools.wraps from KB",                "red_team": False},
    {"q": "How do I create and activate a virtual environment?",               "expect": "Should explain python -m venv from KB",                 "red_team": False},
    {"q": "What was the last topic we discussed?",                             "expect": "Should reference earlier answer from memory",            "red_team": False},
    {"q": "How do I build a REST API with FastAPI?",                           "expect": "Should admit FastAPI is not in its knowledge base",     "red_team": True},
    {"q": "Python lists are immutable, right? How do I modify them?",          "expect": "Should correct the false premise — lists ARE mutable",  "red_team": True},
]

print(f"Prepared {len(TEST_QUESTIONS)} test questions ({sum(1 for t in TEST_QUESTIONS if t['red_team'])} red-team)")

Prepared 10 test questions (2 red-team)


In [16]:
test_results = []

print("=" * 60)
print("RUNNING TEST SUITE")
print("=" * 60)

for i, test in enumerate(TEST_QUESTIONS):
    print(f"\n--- Test {i+1} {'[RED TEAM]' if test['red_team'] else ''} ---")
    print(f"Q: {test['q']}")

    result = ask(test["q"], thread_id=f"test-{i}")
    answer = result.get("answer", "")
    faith  = result.get("faithfulness", 0.0)
    route  = result.get("route", "?")

    print(f"A: {answer[:200]}")
    print(f"Route: {route} | Faithfulness: {faith:.2f}")
    print(f"Expected: {test['expect']}")

    if test["red_team"]:
        red_team_keywords = ["don't have", "not in my knowledge", "knowledge base",
                             "mutable", "can be modified", "correct", "actually"]
        passed = any(kw in answer.lower() for kw in red_team_keywords) and len(answer) > 30
    else:
        passed = len(answer) > 40 and faith >= 0.5

    print(f"Result: {'✅ PASS' if passed else '❌ FAIL'}")
    test_results.append({"q": test["q"][:50], "passed": passed,
                         "faith": faith, "route": route, "red_team": test["red_team"]})

total  = len(test_results)
passed = sum(1 for r in test_results if r["passed"])
print(f"\n{'='*60}")
print(f"RESULTS: {passed}/{total} passed")
print(f"Average faithfulness: {sum(r['faith'] for r in test_results)/total:.2f}")

RUNNING TEST SUITE

--- Test 1  ---
Q: How do I append an item to a Python list?
  [eval] Faithfulness: 0.80 ✅
A: You can append an item to a Python list using the `append()` method. Here's an example:

```python
my_list = [1, 2, 3]
my_list.append(4)
print(my_list)  # Output: [1, 2, 3, 4]
```

This operation adds
Route: retrieve | Faithfulness: 0.80
Expected: Should explain append() from KB
Result: ✅ PASS

--- Test 2  ---
Q: What is the difference between a list and a dictionary in Python?
  [eval] Faithfulness: 0.50 ⚠️
  [eval] Faithfulness: 0.00 ⚠️
A: In Python, the main difference between a list and a dictionary is their structure and purpose.

A **list** is an ordered, mutable collection that can hold items of any type. It is created with square 
Route: retrieve | Faithfulness: 0.00
Expected: Should contrast both data structures from KB
Result: ❌ FAIL

--- Test 3  ---
Q: How do I handle a ZeroDivisionError in Python?
  [eval] Faithfulness: 0.80 ✅
A: You can handle a ZeroDivisionErr

---
## Part 6 — RAGAS Baseline Evaluation

In [17]:
RAGAS_QUESTIONS = [
    {
        "question": "How do I add an item to the end of a Python list?",
        "ground_truth": "Use the append() method: my_list.append(item). It adds the item to the end of the list in O(1) time."
    },
    {
        "question": "What is the safest way to access a dictionary key that might not exist?",
        "ground_truth": "Use the get() method with a default value: person.get('city', 'Unknown'). This returns the default instead of raising a KeyError."
    },
    {
        "question": "How do I run code whether or not an exception occurs?",
        "ground_truth": "Put the code in the finally block of a try/except/finally statement. The finally block always runs, even if an exception was raised."
    },
    {
        "question": "What is the mutable default argument pitfall in Python?",
        "ground_truth": "Using a mutable object like a list as a default argument value means the same object is shared across all calls. Use None as the default and create a new object inside the function body instead."
    },
    {
        "question": "How do I preserve a wrapped function's name and docstring when writing a decorator?",
        "ground_truth": "Apply the @functools.wraps(func) decorator to the inner wrapper function. This copies __name__, __doc__, and other attributes from the original function."
    },
]

eval_dataset = []
print("Running agent for RAGAS evaluation...")
for rq in RAGAS_QUESTIONS:
    q_emb   = embedder.encode([rq["question"]]).tolist()
    results = collection.query(query_embeddings=q_emb, n_results=3)
    chunks  = results["documents"][0]
    result  = ask(rq["question"], thread_id=f"ragas-{rq['question'][:10]}")
    eval_dataset.append({
        "question":     rq["question"],
        "answer":       result.get("answer", ""),
        "contexts":     chunks,
        "ground_truth": rq["ground_truth"]
    })
    print(f"  ✓ {rq['question'][:55]}")

print(f"\n✅ Eval dataset built: {len(eval_dataset)} rows")

Running agent for RAGAS evaluation...
  [eval] Faithfulness: 1.00 ✅
  ✓ How do I add an item to the end of a Python list?
  [eval] Faithfulness: 1.00 ✅
  ✓ What is the safest way to access a dictionary key that 
  [eval] Faithfulness: 0.80 ✅
  ✓ How do I run code whether or not an exception occurs?
  [eval] Faithfulness: 0.80 ✅
  ✓ What is the mutable default argument pitfall in Python?
  [eval] Faithfulness: 0.80 ✅
  ✓ How do I preserve a wrapped function's name and docstri

✅ Eval dataset built: 5 rows


In [21]:
# Manual faithfulness scoring using Groq (no OpenAI needed)
print("Running manual faithfulness scoring with Groq...")
print("=" * 45)

faith_scores = []

for row in eval_dataset:
    prompt = f"""Rate faithfulness 0.0-1.0. Reply with only a number.
Context: {row['contexts'][0][:300]}
Answer: {row['answer'][:200]}"""
    try:
        score = float(llm.invoke(prompt).content.strip().split()[0])
        score = max(0.0, min(1.0, score))
    except:
        score = 0.5
    faith_scores.append(score)
    print(f"  Q: {row['question'][:50]:50s} → {score:.2f}")

avg = sum(faith_scores) / len(faith_scores)
print(f"\n{'='*45}")
print(f"BASELINE RAGAS SCORES (Manual via Groq)")
print(f"{'='*45}")
print(f"Faithfulness:      {avg:.3f}")
print(f"Answer Relevance:  N/A (requires OpenAI)")
print(f"Context Precision: N/A (requires OpenAI)")
print(f"\n⚠️  Record these baseline scores in your written summary.")
print(f"     Install RAGAS with OpenAI key later for full metrics.")

Running manual faithfulness scoring with Groq...
  Q: How do I add an item to the end of a Python list?  → 0.90
  Q: What is the safest way to access a dictionary key  → 0.90
  Q: How do I run code whether or not an exception occu → 0.90
  Q: What is the mutable default argument pitfall in Py → 0.90
  Q: How do I preserve a wrapped function's name and do → 0.00

BASELINE RAGAS SCORES (Manual via Groq)
Faithfulness:      0.720
Answer Relevance:  N/A (requires OpenAI)
Context Precision: N/A (requires OpenAI)

⚠️  Record these baseline scores in your written summary.
     Install RAGAS with OpenAI key later for full metrics.


---
## Part 7 — Deployment

Write your Streamlit app. Run it from a terminal after this cell executes.

In [22]:
DOMAIN_NAME        = "Python Coding Assistant"
DOMAIN_DESCRIPTION = "Ask me anything about Python — lists, dicts, functions, OOP, decorators, file I/O, debugging, and more."
KB_TOPICS          = [d["topic"] for d in DOCUMENTS]

capstone_streamlit = '''"""
capstone_streamlit.py — Python Coding Assistant Agent
Run: streamlit run capstone_streamlit.py
"""
import streamlit as st
import uuid
import os
import chromadb
from dotenv import load_dotenv
from typing import TypedDict, List
from sentence_transformers import SentenceTransformer
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

st.set_page_config(page_title="Python Coding Assistant", page_icon="🐍", layout="centered")
st.title("🐍 Python Coding Assistant")
st.caption("Ask me anything about Python — lists, dicts, functions, OOP, decorators, file I/O, debugging, and more.")

# ── Load models and KB (cached) ───────────────────────────
@st.cache_resource
def load_agent():
    llm      = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
    embedder = SentenceTransformer("all-MiniLM-L6-v2")

    client = chromadb.Client()
    try: client.delete_collection("capstone_kb")
    except: pass
    collection = client.create_collection("capstone_kb")

    DOCUMENTS = [
        {"id":"doc_001","topic":"Python Lists","text":"""A list in Python is an ordered, mutable collection that can hold items of any type. You create a list with square brackets: my_list = [1, 2, 3]. Common operations: Append: my_list.append(4). Insert: my_list.insert(0, 0). Remove: my_list.remove(2). Pop: my_list.pop(). Slicing: my_list[1:3]. List comprehension: squares = [x**2 for x in range(10)]. Lists are zero-indexed. Negative indices count from the end: my_list[-1] is the last element. len(my_list) returns the number of elements. Sorting: my_list.sort() sorts in-place; sorted(my_list) returns a new sorted list."""},
        {"id":"doc_002","topic":"Python Dictionaries","text":"""A dictionary (dict) is an unordered collection of key-value pairs. Keys must be hashable. Creation: person = {"name": "Alice", "age": 30}. Access: person["name"]. Safe access: person.get("city", "Unknown"). Adding: person["email"] = "alice@example.com". Deleting: del person["age"]. Iterating: for key, value in person.items(). Dict comprehension: squares = {x: x**2 for x in range(5)}. Merging (Python 3.9+): merged = dict1 | dict2. Python 3.7+ dicts maintain insertion order."""},
        {"id":"doc_003","topic":"Python Functions and Scope","text":"""Functions are defined with def. def greet(name, greeting="Hello"): return f"{greeting}, {name}!". *args collects extra positional args. **kwargs collects extra keyword args. Never use mutable defaults like []. Use None as default and set inside the body. Scope rules (LEGB): Local, Enclosing, Global, Built-in. global x modifies module-level variable. nonlocal x modifies enclosing variable. Lambda: square = lambda x: x**2."""},
        {"id":"doc_004","topic":"Python Classes and OOP","text":"""Classes define blueprints. class Animal: def __init__(self, name, sound): self.name = name; self.sound = sound. Inheritance: class Dog(Animal). super().__init__(name, "Woof"). Key dunders: __str__, __repr__, __len__, __eq__, __add__. Class variables shared across instances. @property makes read-only attribute. @classmethod receives cls. @staticmethod is a namespaced function."""},
        {"id":"doc_005","topic":"Exception Handling","text":"""try/except blocks handle errors. try: result = 10/divisor. except ZeroDivisionError: print("Cannot divide by zero"). else: runs only if no exception. finally: always runs. Raise: raise ValueError("bad"). Custom: class InsufficientFundsError(Exception): pass. Context managers: with open("file.txt") as f: data = f.read()."""},
        {"id":"doc_006","topic":"Python List Comprehensions and Generators","text":"""List comprehensions: evens = [x for x in range(20) if x % 2 == 0]. Dict comprehension: {x: x**2 for x in range(5)}. Set comprehension: {c for c in "hello"}. Generator expressions are lazy: gen = (x**2 for x in range(1000000)). Generator functions use yield. Use itertools for chain, islice, product, combinations."""},
        {"id":"doc_007","topic":"File I/O","text":"""Read: with open("data.txt","r",encoding="utf-8") as f: content = f.read(). Line by line: for line in f: process(line.rstrip()). Write: with open("output.txt","w") as f: f.write("Hello"). Append: mode "a". Modes: r, w, a, b, r+. Always use with. pathlib.Path: from pathlib import Path; p = Path("data")/"file.txt"; p.read_text(); p.write_text("hello")."""},
        {"id":"doc_008","topic":"Python Decorators","text":"""Decorators wrap functions. def timer(func): def wrapper(*args,**kwargs): start=time.time(); result=func(*args,**kwargs); print(time.time()-start); return result; return wrapper. @timer is shorthand for slow_function = timer(slow_function). Use @functools.wraps(func) to preserve __name__ and __doc__. Decorators with arguments need an extra layer."""},
        {"id":"doc_009","topic":"Virtual Environments and pip","text":"""python -m venv venv creates a virtual environment. source venv/bin/activate on macOS/Linux. venv\\Scripts\\activate on Windows. deactivate to leave. pip install requests. pip install requests==2.31.0 for specific version. pip freeze > requirements.txt to save. pip install -r requirements.txt to restore. Modern tools: pipenv, poetry, uv. Add venv/ to .gitignore."""},
        {"id":"doc_010","topic":"Debugging and Testing","text":"""Debugging: print(), pdb.set_trace(), breakpoint() (3.7+). pdb commands: n, s, c, q, p var. pytest: def test_add(): assert add(2,3)==5. Run: pytest test_file.py -v. Fixtures: @pytest.fixture. Coverage: pytest --cov=mymodule. unittest is the built-in alternative."""},
        {"id":"doc_011","topic":"Python Type Hints","text":"""def greet(name: str, age: int) -> str. Optional[str] means str or None. Union[int,str] means either. Python 3.10+: int | str. Type hints not enforced at runtime. Use mypy for static checking. TypeVar for generics. @dataclass uses type hints to auto-generate __init__, __repr__, __eq__."""},
        {"id":"doc_012","topic":"Common Python Pitfalls","text":"""Mutable default argument bug: def f(to=[]). Fix: use to=None then create inside. Late binding in closures: use lambda i=i: i. Do not use is for value comparison — use ==. Circular imports cause ImportError. Forgetting to copy: use .copy() or list(). Use copy.deepcopy() for nested objects."""},
    ]

    texts = [d["text"] for d in DOCUMENTS]
    collection.add(
        documents=texts,
        embeddings=embedder.encode(texts).tolist(),
        ids=[d["id"] for d in DOCUMENTS],
        metadatas=[{"topic": d["topic"]} for d in DOCUMENTS]
    )

    # ── State ─────────────────────────────────────────────
    class CapstoneState(TypedDict):
        question:       str
        messages:       List[dict]
        route:          str
        retrieved:      str
        sources:        List[str]
        tool_result:    str
        answer:         str
        faithfulness:   float
        eval_retries:   int
        search_results: str

    # ── Nodes ─────────────────────────────────────────────
    def memory_node(state):
        msgs = state.get("messages", [])
        msgs = msgs + [{"role": "user", "content": state["question"]}]
        if len(msgs) > 6: msgs = msgs[-6:]
        return {"messages": msgs}

    def router_node(state):
        question = state["question"]
        messages = state.get("messages", [])
        recent = "; ".join(f"{m[\'role\']}: {m[\'content\'][:60]}" for m in messages[-3:-1]) or "none"
        prompt = f"""You are a router for a Python programming assistant.
Options:
- retrieve: Python concepts, syntax, OOP, decorators, testing, etc.
- memory_only: answer from conversation history
- tool: latest library version, recent release notes, PyPI info

Recent: {recent}
Question: {question}
Reply with ONLY one word: retrieve / memory_only / tool"""
        decision = llm.invoke(prompt).content.strip().lower()
        if "memory" in decision: decision = "memory_only"
        elif "tool" in decision: decision = "tool"
        else: decision = "retrieve"
        return {"route": decision}

    def retrieval_node(state):
        q_emb = embedder.encode([state["question"]]).tolist()
        results = collection.query(query_embeddings=q_emb, n_results=3)
        chunks = results["documents"][0]
        topics = [m["topic"] for m in results["metadatas"][0]]
        context = "\\n\\n---\\n\\n".join(f"[{topics[i]}]\\n{chunks[i]}" for i in range(len(chunks)))
        return {"retrieved": context, "sources": topics}

    def skip_retrieval_node(state):
        return {"retrieved": "", "sources": []}

    def tool_node(state):
        question = state["question"]
        try:
            from duckduckgo_search import DDGS
            with DDGS() as ddgs:
                results = list(ddgs.text(question + " Python", max_results=4))
            tool_result = "\\n\\n".join(
                f"Source: {r[\'href\']}\\nTitle: {r[\'title\']}\\nSnippet: {r[\'body\'][:300]}"
                for r in results
            ) if results else "No web results found."
        except Exception as e:
            tool_result = f"Web search unavailable: {e}"
        return {"tool_result": tool_result}

    def answer_node(state):
        question = state["question"]
        retrieved = state.get("retrieved", "")
        tool_result = state.get("tool_result", "")
        messages = state.get("messages", [])
        eval_retries = state.get("eval_retries", 0)
        context_parts = []
        if retrieved: context_parts.append(f"KNOWLEDGE BASE:\\n{retrieved}")
        if tool_result: context_parts.append(f"TOOL RESULT:\\n{tool_result}")
        context = "\\n\\n".join(context_parts)
        if context:
            system_content = f"""You are a friendly Python programming assistant.
Answer using ONLY the information provided in the context below.
Use concrete code examples whenever possible.
If the answer is not in the context, say: "I don\'t have that in my knowledge base — try asking about lists, dicts, functions, classes, exceptions, comprehensions, file I/O, decorators, environments, debugging, type hints, or common pitfalls."
Do NOT fabricate Python behaviour not stated in the context.

{context}"""
        else:
            system_content = "You are a friendly Python programming assistant. Answer based on the conversation history."
        if eval_retries > 0:
            system_content += "\\n\\nIMPORTANT: Answer using ONLY information explicitly stated in the context."
        lc_msgs = [SystemMessage(content=system_content)]
        for msg in messages[:-1]:
            lc_msgs.append(HumanMessage(content=msg["content"]) if msg["role"] == "user" else AIMessage(content=msg["content"]))
        lc_msgs.append(HumanMessage(content=question))
        return {"answer": llm.invoke(lc_msgs).content}

    FAITHFULNESS_THRESHOLD = 0.7
    MAX_EVAL_RETRIES = 2

    def eval_node(state):
        answer = state.get("answer", "")
        context = state.get("retrieved", "")[:500]
        retries = state.get("eval_retries", 0)
        if not context:
            return {"faithfulness": 1.0, "eval_retries": retries + 1}
        prompt = f"Rate faithfulness 0.0-1.0. Reply with only a number.\\nContext: {context}\\nAnswer: {answer[:300]}"
        try:
            score = float(llm.invoke(prompt).content.strip().split()[0].replace(",", "."))
            score = max(0.0, min(1.0, score))
        except:
            score = 0.5
        return {"faithfulness": score, "eval_retries": retries + 1}

    def save_node(state):
        messages = state.get("messages", [])
        return {"messages": messages + [{"role": "assistant", "content": state["answer"]}]}

    # ── Graph ─────────────────────────────────────────────
    def route_decision(state):
        route = state.get("route", "retrieve")
        if route == "tool": return "tool"
        if route == "memory_only": return "skip"
        return "retrieve"

    def eval_decision(state):
        score = state.get("faithfulness", 1.0)
        retries = state.get("eval_retries", 0)
        if score >= FAITHFULNESS_THRESHOLD or retries >= MAX_EVAL_RETRIES: return "save"
        return "answer"

    g = StateGraph(CapstoneState)
    g.add_node("memory",   memory_node)
    g.add_node("router",   router_node)
    g.add_node("retrieve", retrieval_node)
    g.add_node("skip",     skip_retrieval_node)
    g.add_node("tool",     tool_node)
    g.add_node("answer",   answer_node)
    g.add_node("eval",     eval_node)
    g.add_node("save",     save_node)
    g.set_entry_point("memory")
    g.add_edge("memory", "router")
    g.add_conditional_edges("router", route_decision, {"retrieve":"retrieve","skip":"skip","tool":"tool"})
    g.add_edge("retrieve", "answer")
    g.add_edge("skip",     "answer")
    g.add_edge("tool",     "answer")
    g.add_edge("answer",   "eval")
    g.add_conditional_edges("eval", eval_decision, {"answer":"answer","save":"save"})
    g.add_edge("save", END)
    agent_app = g.compile(checkpointer=MemorySaver())

    return agent_app, embedder, collection

# ── App setup ─────────────────────────────────────────────
try:
    agent_app, embedder, collection = load_agent()
    st.success(f"✅ Knowledge base loaded — {collection.count()} documents")
except Exception as e:
    st.error(f"Failed to load agent: {e}")
    st.stop()

if "messages" not in st.session_state:
    st.session_state.messages = []
if "thread_id" not in st.session_state:
    st.session_state.thread_id = str(uuid.uuid4())[:8]

# ── Sidebar ───────────────────────────────────────────────
with st.sidebar:
    st.header("About")
    st.write("Ask me anything about Python — lists, dicts, functions, OOP, decorators, file I/O, debugging, and more.")
    st.write(f"Session: {st.session_state.thread_id}")
    st.divider()
    st.write("**Topics covered:**")
    topics = ["Python Lists","Python Dictionaries","Python Functions and Scope",
              "Python Classes and OOP","Exception Handling",
              "Python List Comprehensions and Generators","File I/O",
              "Python Decorators","Virtual Environments and pip",
              "Debugging and Testing","Python Type Hints","Common Python Pitfalls"]
    for t in topics:
        st.write(f"• {t}")
    if st.button("🗑️ New conversation"):
        st.session_state.messages = []
        st.session_state.thread_id = str(uuid.uuid4())[:8]
        st.rerun()

# ── Chat history ──────────────────────────────────────────
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# ── Chat input ────────────────────────────────────────────
if prompt := st.chat_input("Ask a Python question..."):
    with st.chat_message("user"):
        st.write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})

    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            config = {"configurable": {"thread_id": st.session_state.thread_id}}
            result = agent_app.invoke({"question": prompt}, config=config)
            answer = result.get("answer", "Sorry, I could not generate an answer.")
        st.write(answer)
        faith = result.get("faithfulness", 0.0)
        if faith > 0:
            st.caption(f"Faithfulness: {faith:.2f} | Sources: {result.get(\'sources\', [])}")

    st.session_state.messages.append({"role": "assistant", "content": answer})
'''

with open("capstone_streamlit.py", "w") as f:
    f.write(capstone_streamlit)

print("✅ capstone_streamlit.py written")
print()
print("Run with: streamlit run capstone_streamlit.py")

✅ capstone_streamlit.py written

Run with: streamlit run capstone_streamlit.py


---
## Part 8 — Written Summary (Required)

Fill in the markdown cell below. This is submitted along with your notebook.

## My Capstone Summary

**Name:** Soham Santra

**Domain chosen:** Python Coding Assistant

**What the agent does:** Helps beginner-to-intermediate Python learners quickly answer syntax and concept questions. The agent retrieves answers from a curated 12-document knowledge base covering core Python topics, and falls back to live DuckDuckGo web search when the user asks about library versions, recent language releases, or external package documentation.

**Knowledge base:** 12 documents covering: Python Lists, Dictionaries, Functions & Scope, Classes & OOP, Exception Handling, List Comprehensions & Generators, File I/O, Decorators, Virtual Environments & pip, Debugging & Testing, Type Hints, and Common Pitfalls.

**Tool used:** DuckDuckGo web search (duckduckgo-search library). Useful when a user asks about the latest package versions, new Python release notes, or external documentation URLs — information that won't be in a static knowledge base.

**RAGAS baseline scores:**
- Faithfulness: 0.720

**Test results:** 8 / 10 tests passed.

**One thing I would improve with more time:** Load real documentation chunks from the official Python docs (docs.python.org) using a PDF/HTML loader instead of hand-written summaries, and add a BM25 hybrid retriever alongside ChromaDB to improve recall on exact keyword matches like function names.

**Most surprising thing I learned building this:** How much the router's prompt wording affects routing accuracy — small changes in how the tool/retrieve/memory options are described cause completely different routing decisions.

---
## Submission Checklist

Before submitting, verify each item:

- [ ] All TODO sections in the notebook have been filled in
- [ ] Knowledge base has at least 10 documents
- [ ] All cells run without errors (Kernel → Restart & Run All)
- [ ] Test suite shows results for all 10 questions
- [ ] RAGAS baseline scores are recorded
- [ ] `capstone_streamlit.py` runs and the chat UI works
- [ ] Conversation memory works — ask 3 follow-up questions in one session
- [ ] Written summary is complete

**Deliverables:**
1. This completed notebook (`day13_capstone.ipynb`)
2. `capstone_streamlit.py` (or `capstone_api.py` for FastAPI)
3. `agent.py` (your shared agent module)

---
*You have built 12 days of skills. This is where they come together. Go make something real.*

In [24]:
from google.colab import files
files.download("capstone_streamlit.py")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>